In [66]:
#
import requests
import pandas as pd 
pd.set_option('display.max_rows', None)
from IPython.display import HTML

#
import mysql.connector
from mysql.connector import Error

#
import numpy as np

#
import os 
from dotenv import load_dotenv
load_dotenv()
password_sql = os.getenv("PASS_SQL")

import json

# EXTRACCIÓN DE DATOS EN DEEZER Y LAST.FM

In [ ]:
# Función genérica para extraer las API 
def extraccion_API(endpoint, verbose=True):

    try:
        # Endpoint como argumento de función
        datos = requests.get(endpoint)
        if datos.status_code == 200:
            # Se permite print detallado
            if verbose: 
                print ("API conectada correctamente")
            # Se convierte datos a JSON para que sea manejable por el resto de funciones
            datos_json = datos.json()
            return datos_json
        else:
            # En caso de error, se imprime fallo 
            print (f"Error de conexión en API {endpoint}: {datos.status_code}")

    # Error de conexión        
    except requests.exceptions.ConnectionError as CnxE:
        print (CnxE)

    # Error de tiempo de espera
    except requests.exceptions.Timeout as TO:
        print (TO)
    
    # Cubrimos el resto de errores
    except requests.exceptions.RequestException as e:
        print (e) 

In [ ]:
# Lista los ID de los cantantes o grupos que se van a extraer
id_artistas = [12246, 160, 145, 564, 75491, 75798, 290, 483, 10803980, 1538640, 892, 10583405, 412, 13, 4050205, 384236, 119, 5620251, 259, 5962948, 196, 485, 425, 315929, 2446, 1755, 180, 98, 10977, 1434]

In [ ]:
# Función para extraer los datos de artistas de Deezer
def extraer_artistas(id_artistas):

    # Se define lista vacía y el contador de verbose
    lista_artistas = []
    contador = 0
    # Se recorre un endpoint por artista
    for artistas in id_artistas:
        endpoint = f"https://api.deezer.com/artist/{artistas}"

        # Se lanza la función para extraer las API
        datos_artistas = extraccion_API(endpoint, verbose=False)
        contador += 1
        # Diccionario con los datos necesarios
        diccionario_artista = {
            "id_artista": datos_artistas["id"],
            "nombre": datos_artistas["name"]
            }
        lista_artistas.append(diccionario_artista)

    # Imprime el número de apis correctas
    print(f"{contador} APIs extraídas correctamente")
    # Unimos la lista de DF
    df_final = pd.DataFrame(lista_artistas)
    return df_final

In [23]:
df_artistas = extraer_artistas(id_artistas)
HTML(df_artistas.to_html(index=False))

30 APIs extraídas correctamente


id_artista,nombre
12246,Taylor Swift
160,Shakira
145,Beyoncé
564,Rihanna
75491,Lady Gaga
75798,Adele
290,Madonna
483,Britney Spears
10803980,BLACKPINK
1538640,Little Mix


In [45]:
# Función para extraer los datos de las canciones de Deezer
def extraer_canciones(id_artistas):

    # Lista vacía para el DF y contador
    lista_canciones = []
    contador = 0
    # Se recorre un endpoint por artista
    for artistas in id_artistas:
        endpoint = f"https://api.deezer.com/artist/{artistas}/top?limit=50"
        
        # Se lanza la función para extraer las API
        datos_canciones = extraccion_API(endpoint, verbose = False)
        contador += 1

        # Bucle para recoger los datos que necesitamos del JSON
        for cancion in datos_canciones["data"]:
            id_cancion = cancion["id"]
            id_artista = cancion["artist"]["id"]
            id_album = cancion["album"]["id"]
            titulo = cancion["title"]
            duracion = cancion["duration"]
            ranking = cancion["rank"]
            if len(cancion["contributors"]) == 1:
                colaboradores = False
            else:
                colaboradores = True

            # Diccionario creado con los datos extraidos del JSON
            diccionario_cancion = {
                    "id_cancion": id_cancion,
                    "id_artista": id_artista,
                    "id_album": id_album,
                    "title": titulo,
                    "duration": duracion,
                    "rank": ranking,
                    "contributors_bool": colaboradores
                }         
                        
            # Se añade el diccionario a la lista
            lista_canciones.append(diccionario_cancion)
    
    # Imprime el número de apis correctas
    print(f"{contador} APIs extraídas correctamente")        
    df_final = pd.DataFrame(lista_canciones)
    return df_final, lista_canciones

In [46]:
df_canciones ,  lista_canciones = extraer_canciones(id_artistas) 
df_canciones.to_csv("listado_de_canciones.csv", index=False)

30 APIs extraídas correctamente


In [ ]:
# Funcion para comprobar que los artistas elegidos tienen 50 canciones
def conteo_artistas(id_artistas):

    # Diccionario vacío para mostrar conteo
    diccionario_artistas = {}
    # Se recorren las APIs
    for artistas in id_artistas:
        endpoint = f"https://api.deezer.com/artist/{artistas}/top?limit=50"
        datos = extraccion_API(endpoint, verbose=False)
        
        # Enfrenta artistas con el número de canciones en la API
        diccionario_artistas[artistas] = len(datos["data"])
    return diccionario_artistas

In [39]:
diccionario_artistas = conteo_artistas(id_artistas)
diccionario_artistas

{12246: 50,
 160: 50,
 145: 50,
 564: 50,
 75491: 50,
 75798: 50,
 290: 50,
 483: 50,
 10803980: 50,
 1538640: 50,
 892: 50,
 10583405: 50,
 412: 50,
 13: 50,
 4050205: 50,
 384236: 50,
 119: 50,
 5620251: 50,
 259: 50,
 5962948: 50,
 196: 50,
 485: 50,
 425: 50,
 315929: 50,
 2446: 50,
 1755: 50,
 180: 50,
 98: 50,
 10977: 50,
 1434: 50}

In [40]:
# Función para extraer los géneros de Deezer
def extraer_genero():

    # Endpoint único
    endpoint = "https://api.deezer.com/genre/"
    try:
        datos_genero = extraccion_API(endpoint, verbose = True)
        df_genero = pd.DataFrame(datos_genero["data"])
        df_final = df_genero[["id","name"]]                       
        return df_final
    except:
        print ("error")

In [41]:
df_genero = extraer_genero()
HTML(df_genero.to_html(index=False))

API conectada correctamente


id,name
0,Todos
132,Pop
116,Rap/Hip Hop
122,Reggaeton
152,Rock
113,Dance
165,R&B
85,Alternativo
106,Electro
466,Folk


In [49]:
def extraer_album():
    album_ids_unicos = set()

    for cancion in lista_canciones:
        album_ids_unicos.add(cancion["id_album"])

    contador = 0
    lista_albumes = []
    for album_id in album_ids_unicos:
        endpoint = f"https://api.deezer.com/album/{album_id}"
        datos_albumes = extraccion_API(endpoint, verbose = False)
        contador += 1
        
        diccionario_albumes = {
            "id_album": datos_albumes["id"],
            "id_artista": datos_albumes["artist"]["id"],
            "id_genre": datos_albumes["genre_id"],
            "titulo": datos_albumes["title"],
            "n_canciones": datos_albumes["nb_tracks"],
            "fecha_lanzamiento": datos_albumes["release_date"]
        }     

        
        lista_albumes.append(diccionario_albumes)
    
    print(f"{contador} APIs extraídas correctamente") 
    df_final = pd.DataFrame(lista_albumes)
    return df_final
        


In [50]:
df_albumes = extraer_album()
df_albumes.to_csv("listado_de_albumes.csv", index=False)

611 APIs extraídas correctamente


In [55]:
def extraer_last_fm():

    lista_artistas = []
    contador = 0
    df_artistas["nombre"].str.replace(" ", "+")
    for nombre in df_artistas["nombre"]:
        endpoint = f"https://ws.audioscrobbler.com/2.0/?method=artist.getInfo&artist={nombre}&api_key=68eddfdc6b072ad56527a4199d979038&format=json"
        datos_last_fm = extraccion_API(endpoint, verbose=False)
        contador += 1
    
        diccionario_artistas = {
            "nombre" : datos_last_fm["artist"]["name"],
            "oyentes" : datos_last_fm["artist"]["stats"]["listeners"],
            "reproducciones" : datos_last_fm["artist"]["stats"]["playcount"],
            "biografia" : datos_last_fm["artist"]["bio"]["summary"]
        }
            
        
        lista_artistas.append(diccionario_artistas)

    df_final = pd.DataFrame(lista_artistas)
    df_final["biografia"] = df_final["biografia"].str.replace("\n", " ")
    print(f"{contador} APIs extraídas correctamente") 
    return df_final, lista_artistas
        


In [57]:
df_artistas_fm, lista_artistas_fm = extraer_last_fm()
df_artistas_fm.to_csv("listado_de_artistas_fm.csv", index=False)

30 APIs extraídas correctamente


# INSERCIÓN EN MYSQL

In [67]:
def conectar_mysql(host="127.0.0.1", user="root", password=password_sql, database=None):
    try:
        cnx = mysql.connector.connect(
            host=host,
            user=user,
            password=password,
            database=database        # si no le pasas base de datos, se conecta al servidor solo
        )
        print("Conexión exitosa")
        return cnx                      # con este return guarda la conexion y puedo usar esta conexion despues 
    except Error as e:
        print(f"Error al conectar: {e}")

In [68]:
conexion = conectar_mysql()

Conexión exitosa


In [69]:
nombre_bd = "proyecto_music_stream_team1"

In [70]:
# 
def crear_basededatos(nombre_bd):
   
    try:
        # 
        with conexion.cursor() as cursor:
            query = f"CREATE DATABASE IF NOT EXISTS {nombre_bd}"
            # 
            cursor.execute(query)
            print ("Query exitosa")
 
    # 
    except Error as e:
        print (f"Error creando base de datos: {e}")

In [71]:
crear_basededatos(nombre_bd)

Query exitosa


In [72]:
# para crear tablas
def crear_tablas_genericas(nombre_bd, nombre_tabla, tabla_esquema):
   
    try:
        # 
        with conexion.cursor() as cursor:
            cursor.execute(f"USE {nombre_bd};")
            # 
            query = f''' CREATE TABLE IF NOT EXISTS {nombre_tabla} ({tabla_esquema});'''
            #
            cursor.execute(query)
            print ("Query creación exitosa")
   
    #
    except Error as e:
        print (f"Error creando tabla: {e}")

In [73]:
tabla_artista = 'artista'
tabla_genero_musical = 'genero_musical'
tabla_canciones = 'canciones'
tabla_album = 'album'

In [74]:
esquema_artista = '''id_artista INT PRIMARY KEY,
   nombre VARCHAR(30) NOT NULL,
   oyentes INT,
   reproducciones INT,
   biografia VARCHAR(1000) NOT NULL,
   genero VARCHAR(10) NOT NULL
   '''

In [75]:
esquema_genero_musical = '''id_genero INT PRIMARY KEY, 
nombre VARCHAR(40)'''

In [76]:
esquema_canciones = '''id_cancion INT PRIMARY KEY,
id_artista INT NOT NULL,
id_album INT NOT NULL,
titulo VARCHAR(200) NOT NULL,
duracion INT,
ranking_lista INT,
colaboraciones BOOLEAN
'''

In [77]:
esquema_album = ''' id_album INT PRIMARY KEY,
id_artista INT,
id_genero INT,
titulo VARCHAR(200) NOT NULL,
numero_canciones INT,
fecha_lanzamiento DATE
'''

In [78]:
crear_tablas_genericas(nombre_bd,tabla_artista, esquema_artista)

Query creación exitosa


In [79]:
crear_tablas_genericas(nombre_bd, tabla_genero_musical, esquema_genero_musical)

Query creación exitosa


In [80]:
crear_tablas_genericas(nombre_bd,tabla_canciones, esquema_canciones)

Query creación exitosa


In [81]:
crear_tablas_genericas(nombre_bd,tabla_album, esquema_album)

Query creación exitosa
